# Set Default lakehouse

In [2]:
%%configure -f
{
  "defaultLakehouse": {
    "name": {
      "variableName": "$(/**/vl_sales/lakehouse_name)" 
    },
    "id": {
      "variableName": "$(/**/vl_sales/lakehouse_id)"
    },
    "workspaceId": {
      "variableName": "$(/**/vl_sales/workspace_id)"
    }
  }
}

StatementMeta(, 77c2b1bf-4510-42cf-bd74-62c4924b5672, -1, Finished, Available, Finished, True)

In [4]:
# refer https://blog.fabric.microsoft.com/en-US/blog/variable-library-support-in-notebook-now-generally-available/

lakehouse_name=notebookutils.variableLibrary.get("$(/**/vl_sales/lakehouse_name)")
lakehouse_id=notebookutils.variableLibrary.get("$(/**/vl_sales/lakehouse_id)")
workspace_id=notebookutils.variableLibrary.get("$(/**/vl_sales/workspace_id)")

StatementMeta(, 77c2b1bf-4510-42cf-bd74-62c4924b5672, 4, Finished, Available, Finished, False)

In [30]:
lakehouse_name

StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 31, Finished, Available, Finished, False)

'lh_sales'

In [18]:
#%%sql
#DROP SCHEMA metadata1

StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 19, Finished, Available, Finished, False)

# Create Schema

In [31]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.metadata
""")

StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 32, Finished, Available, Finished, False)

DataFrame[]

# Create Table

In [32]:
spark.sql(f"""
DROP TABLE IF EXISTS {lakehouse_name}.metadata.SourceTargetSystem
""")

StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 33, Finished, Available, Finished, False)

DataFrame[]

In [33]:
spark.sql(f"""
CREATE TABLE {lakehouse_name}.metadata.SourceTargetSystem (
    SourceConnectionString STRING,
    SourceSystemType       STRING NOT NULL,
    SourceSystemName       STRING NOT NULL,
    TargetSystemType       STRING NOT NULL,
    TargetFolderOrSchemaName       STRING NOT NULL,
    TargetFileOrTableName       STRING NOT NULL,
    CreatedDate            TIMESTAMP NOT NULL,
    ModifiedDate           TIMESTAMP NOT NULL,
    CreatedBy              STRING NOT NULL
)
USING DELTA
""")

StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 34, Finished, Available, Finished, False)

DataFrame[]

# Insert Data

In [34]:
# spark.sql(f"""
# INSERT INTO {lakehouse_name}.metadata.SourceTargetSystem
# SELECT
#     'rritec/Microsoft-Fabric/refs/heads/main/Labdata/emp.csv',
#     'csv',
#     'GITHUB',
#     'Lakehouse',
#     'data',
#     'emp.csv',
#     current_timestamp(),
#     current_timestamp(),
#     current_user()
# """)

StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 35, Finished, Available, Finished, False)

DataFrame[]

In [36]:
# spark.sql(f"""
# select * from {lakehouse_name}.metadata.SourceTargetSystem
# """).show(truncate=False)


StatementMeta(, eb045b16-7603-418a-bbda-44519af5965b, 37, Finished, Available, Finished, False)

+-------------------------------------------------------+----------------+----------------+----------------+------------------------+---------------------+-------------------------+-------------------------+--------------------+
|SourceConnectionString                                 |SourceSystemType|SourceSystemName|TargetSystemType|TargetFolderOrSchemaName|TargetFileOrTableName|CreatedDate              |ModifiedDate             |CreatedBy           |
+-------------------------------------------------------+----------------+----------------+----------------+------------------------+---------------------+-------------------------+-------------------------+--------------------+
|rritec/Microsoft-Fabric/refs/heads/main/Labdata/emp.csv|csv             |GITHUB          |Lakehouse       |data                    |emp.csv              |2026-03-04 01:47:02.25395|2026-03-04 01:47:02.25395|trusted-service-user|
+-------------------------------------------------------+----------------+----------

In [9]:
spark.sql(f"""
MERGE INTO {lakehouse_name}.metadata.SourceTargetSystem AS tgt
USING (
    SELECT
        'rritec/Microsoft-Fabric/refs/heads/main/Labdata/emp.csv' AS SourceConnectionString,
        'csv' AS SourceSystemType,
        'GITHUB' AS SourceSystemName,
        'Lakehouse' AS TargetSystemType,
        'data' AS TargetFolderOrSchemaName,
        'emp.csv' AS TargetFileOrTableName,
        current_timestamp() AS CreatedDate,
        current_timestamp() AS ModifiedDate,
        current_user() AS CreatedBy

    UNION ALL

    SELECT
        'rritec/Microsoft-Fabric/refs/heads/main/Labdata/dept.csv',
        'csv',
        'GITHUB',
        'Lakehouse',
        'data',
        'dept.csv',
        current_timestamp(),
        current_timestamp(),
        current_user()
) src

ON tgt.SourceConnectionString = src.SourceConnectionString
AND tgt.TargetFileOrTableName = src.TargetFileOrTableName

WHEN NOT MATCHED THEN
INSERT (
    SourceConnectionString,
    SourceSystemType,
    SourceSystemName,
    TargetSystemType,
    TargetFolderOrSchemaName,
    TargetFileOrTableName,
    CreatedDate,
    ModifiedDate,
    CreatedBy
)
VALUES (
    src.SourceConnectionString,
    src.SourceSystemType,
    src.SourceSystemName,
    src.TargetSystemType,
    src.TargetFolderOrSchemaName,
    src.TargetFileOrTableName,
    src.CreatedDate,
    src.ModifiedDate,
    src.CreatedBy
)
""")

StatementMeta(, 77c2b1bf-4510-42cf-bd74-62c4924b5672, 9, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [10]:
spark.sql(f"""
select * from {lakehouse_name}.metadata.SourceTargetSystem
""").show(truncate=False)


StatementMeta(, 77c2b1bf-4510-42cf-bd74-62c4924b5672, 10, Finished, Available, Finished, False)

+--------------------------------------------------------+----------------+----------------+----------------+------------------------+---------------------+--------------------------+--------------------------+--------------------+
|SourceConnectionString                                  |SourceSystemType|SourceSystemName|TargetSystemType|TargetFolderOrSchemaName|TargetFileOrTableName|CreatedDate               |ModifiedDate              |CreatedBy           |
+--------------------------------------------------------+----------------+----------------+----------------+------------------------+---------------------+--------------------------+--------------------------+--------------------+
|rritec/Microsoft-Fabric/refs/heads/main/Labdata/emp.csv |csv             |GITHUB          |Lakehouse       |data                    |emp.csv              |2026-03-04 02:08:08.640906|2026-03-04 02:08:08.640906|trusted-service-user|
|rritec/Microsoft-Fabric/refs/heads/main/Labdata/dept.csv|csv           